In [ ]:
from IPython.display import IFrame
IFrame("https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones", 900,500)

In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo('XOEN9W05_4A')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import keras_tuner as kt
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Normalize dataInertial Signals
def load_signals(subset='train'):
    signals = [
        'body_acc_x',
        'body_acc_y',
        'body_acc_z',
        'body_gyro_x',
        'body_gyro_y',
        'body_gyro_z',
        'total_acc_x',
        'total_acc_y',
        'total_acc_z'
    ]
    data = []
    for signal in signals:
        filename = f'UCI_HAR_Dataset/{subset}/Inertial Signals/{signal}_{subset}.txt'
        df = pd.read_csv(filename, sep=r'\s+', header=None)
        data.append(df.values)
    # Stack signals to form (samples, time_steps, features)
    X = np.stack(data, axis=2)  # (samples, 128, 9)
    return X

# Normalize datatrain  test signals
X_train = load_signals('train')  # shape (7352, 128, 9)
y_train = pd.read_csv('UCI_HAR_Dataset/train/y_train.txt', header=None)
X_test = load_signals('test')    # shape (2947, 128, 9)
y_test = pd.read_csv('UCI_HAR_Dataset/test/y_test.txt', header=None)

# Normalize datatrain  test
X = np.concatenate((X_train, X_test), axis=0)  # shape (10299, 128, 9)
y = pd.concat([y_train, y_test], axis=0).values.ravel() - 1  # Normalize datalabels  1-6  0-5
y = to_categorical(y)  # Normalize datacategorical

# Normalize dataX 
print(f"X shape before reshape: {X.shape}")  # Normalize data(10299, 128, 9)

# Normalize datatrain  validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Example
print("X_train sample (5 rows):")
print(X_train[:5])

print("\ny_train sample (5 rows):")
print(y_train[:5])

In [ ]:
# Create CNN model-LSTM
def build_cnn_lstm_model(hp):
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Conv1D(filters=hp.Int('filters', 32, 128, step=32), kernel_size=3, activation='relu', input_shape=(128, 9)))
    model.add(tf.keras.layers.MaxPooling1D(pool_size=2))
    model.add(tf.keras.layers.Dropout(0.5))
    model.add(tf.keras.layers.LSTM(hp.Int('units', 32, 128, step=32)))
    model.add(tf.keras.layers.Dense(6, activation='softmax'))  # Normalize data6 classes

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Use Random Search via KerasTuner  Grid Search
tuner_cnn_lstm = kt.RandomSearch(
    build_cnn_lstm_model,
    objective='val_accuracy',
    max_trials=5,  # Define number of trials (different tests)
    directory='random_search', 
    project_name='cnn_lstm_har'
)

# Run the search
tuner_cnn_lstm.search(X_train, y_train, epochs=5, validation_data=(X_val, y_val))

# Display best hyperparameters
best_hps_cnn_lstm = tuner_cnn_lstm.get_best_hyperparameters(num_trials=1)[0]

print(f'Best number of filters: {best_hps_cnn_lstm.get("filters")}')
print(f'Best number of LSTM units: {best_hps_cnn_lstm.get("units")}')

# Build and train the best model with optimized hyperparameters
best_cnn_lstm_model = tuner_cnn_lstm.hypermodel.build(best_hps_cnn_lstm)
best_cnn_lstm_model.fit(X_train, y_train, epochs=5, validation_data=(X_val, y_val))

In [ ]:
# Predict results from X_val  CNN-LSTM 
y_pred_cnn_lstm = best_cnn_lstm_model.predict(X_val)
y_pred_cnn_lstm_classes = y_pred_cnn_lstm.argmax(axis=1)  # Normalize dataclass

# Create Confusion Matrix
y_true_classes = y_val.argmax(axis=1)  # Normalize dataclass
cm_cnn_lstm = confusion_matrix(y_true_classes, y_pred_cnn_lstm_classes)

# Display Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm_cnn_lstm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix for CNN-LSTM Model')
plt.show()

# Calculate Accuracy, Precision, Recall, and F1-score
accuracy_cnn_lstm = accuracy_score(y_true_classes, y_pred_cnn_lstm_classes)
precision_cnn_lstm = precision_score(y_true_classes, y_pred_cnn_lstm_classes, average='weighted')
recall_cnn_lstm = recall_score(y_true_classes, y_pred_cnn_lstm_classes, average='weighted')
f1_cnn_lstm = f1_score(y_true_classes, y_pred_cnn_lstm_classes, average='weighted')

# Display Accuracy, Precision, Recall, and F1-score
print(f'Accuracy: {accuracy_cnn_lstm:.4f}')
print(f'Precision: {precision_cnn_lstm:.4f}')
print(f'Recall: {recall_cnn_lstm:.4f}')
print(f'F1-score: {f1_cnn_lstm:.4f}')